# Colab Part A: hybrid TimesFM + LSTM heads on CSE, Binance and Yahoo

**J26-SE-325, Component 1.** The TAF architecture: a frozen TimesFM 2.5 base forecasts
returns, and an LSTM/MLP residual head corrects it from technical indicators, per-tier
sentiment, point-in-time fundamentals and the asset class. This notebook trains one head per
horizon (5 and 21 trading days) and exports them (~a few hundred KB) for the local service.

**Run Part D first** (`colab_fundamentals_backfill.ipynb`): the fundamentals inputs come from
its parquet. Without it the fundamentals columns are zero and `has_fundamentals = 0`.

**Runtime:** T4 GPU. **Resumable:** bars, features and each finished head are written to
Drive; after a disconnect run all cells again and finished stages are skipped.

(`colab_finetune.ipynb` remains for the optional LoRA adaptation of the base model itself.)

## TimesFM 3.0 and its licence

The heads are trained over **TimesFM 3.0**, the latest release (`google/timesfm-3.0-pytorch`, 330M
parameters). Its weights are distributed under the *TimesFM non-commercial licence*: research use is
fine, **commercial or production use of self-hosted weights is not permitted** (Google Cloud's
BigQuery ML is the licensed commercial route). TimesFM 2.5 weights remain Apache-2.0. Set
`TIMESFM_GENERATION = "2.5"` to train over 2.5 instead. The generation is recorded in each head's
`meta.json`, and serving always rebuilds the base a head was trained on.

## CSE data

CSE names are trained as their own `cse_equity` class. The CSE chart endpoint returns about **one
year** of daily bars and nothing longer, so CSE results come from a single year and a holdout of a
few dozen sessions per name; the results table flags this on the row. To train on more than a year,
run `python -m data.cse_history` daily on the laptop (it merges each fetch into
`artifacts/cse_history/`), zip that folder to Drive as `j26_cse_history/`, and the notebook picks it
up.

## Train/holdout split (changed)

The first run cut every asset class at ONE global date. CSE bars cover only the last year, which
starts after that date, so **every CSE row was holdout and none trained** (the hybrid scored 21.7% worse
than its base on CSE at h5). The split is now **per class** (`--split per-class`): each class is cut at the
80th percentile of its own dates, so CSE gets training rows and a holdout of its last ~50 sessions.
The cost is stated, not hidden: CSE training rows fall in a window where US and crypto are held out, so
the shared head sees that period's CSE moves while those classes are scored on it (CSE and US are
different markets, so the leak is small). `meta.json` records `split` and the cutoff per class.


In [ ]:
# === BOOTSTRAP (same contract as colab_finetune.ipynb) ================================
# Clone or update the branch, chdir into the component, install PIP_PKGS. Safe to re-run.
REPO_URL = "https://github.com/SE-Y4S1/J26-SE-325.git"
BRANCH = "Nivakaran"  # the component is NOT on main
SUBDIR = "backend/Portfolio-Optimization"
PIP_PKGS = "timesfm[torch]==3.0.2 pandas-ta-openbb yfinance pypdf pyyaml requests pydantic pydantic-settings"

import importlib, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
OWN_PACKAGES = {"agent", "configs", "data", "evaluation", "experiments", "features",
                "forecasting", "fundamentals", "news", "optimization", "decision", "service"}


def find_root(start):
    for base in (start, *start.parents):
        if (base / "configs/universe.yaml").exists() and (base / "fundamentals").exists():
            return base
    return None


# In Colab always fetch: a re-run starts INSIDE the checkout, so find_root
# succeeded and pushed fixes never reached a live runtime.
root = None if IN_COLAB else find_root(Path.cwd())
if root is None:
    if not IN_COLAB:
        raise SystemExit("cd into backend/Portfolio-Optimization first.")
    checkout = Path("/content/J26-SE-325")
    if (checkout / SUBDIR).exists():
        if subprocess.run(["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", BRANCH]).returncode == 0:
            subprocess.run(["git", "-C", str(checkout), "reset", "--hard", "FETCH_HEAD"])
    else:
        if subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)]).returncode:
            from getpass import getpass
            token = getpass("Clone failed. GitHub token (repo scope), blank to abort: ").strip()
            if not token:
                raise SystemExit("Cannot reach " + REPO_URL)
            subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                            REPO_URL.replace("https://", "https://" + token + "@"), str(checkout)], check=True)
    root = checkout / SUBDIR

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
for _name in [n for n in sys.modules if n.split(".")[0] in OWN_PACKAGES]:
    del sys.modules[_name]
importlib.invalidate_caches()
if IN_COLAB and PIP_PKGS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PIP_PKGS.split(), check=True)
print("OK root=" + str(root))

In [ ]:
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"

# The adapter registry reads artifacts/env_report.json (gitignored), so write it here.
from forecasting.env_report import write_env_report

report = write_env_report()
assert report["optional_forecasters"]["timesfm"]["available"], report["optional_forecasters"]["timesfm"]
assert report["optional_forecasters"]["timesfm3"]["available"], report["optional_forecasters"]["timesfm3"]

In [ ]:
from pathlib import Path

# TimesFM base for the heads. "3.0" is the latest release (Aug 2026). Its weights are under a
# NON-COMMERCIAL, non-production licence: fine for this research run, but a head trained on 3.0 is
# served over 3.0 weights, so it must not go into a commercial deployment. "2.5" is Apache-2.0.
TIMESFM_GENERATION = "3.0"   # "3.0" | "2.5" | "auto"

OUT = Path("artifacts/hybrid_training")
BACKFILL = None
CSE_HISTORY = None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # A NEW folder. j26_hybrid holds a bars.parquet written by a Laya smoke run (20 CSE names), which
    # the first full run silently reused: only 20 of ~270 CSE names were trained. The trainer now
    # records its settings in the folder and refuses one built under different settings.
    OUT = Path("/content/drive/MyDrive/j26_hybrid_v2")
    candidate = Path("/content/drive/MyDrive/j26_se_325_backfill/fundamentals_backfill.parquet")
    BACKFILL = candidate if candidate.exists() else None
    history = Path("/content/drive/MyDrive/j26_cse_history")
    CSE_HISTORY = history if history.exists() else None
OUT.mkdir(parents=True, exist_ok=True)
print("out:", OUT, "| TimesFM:", TIMESFM_GENERATION)
print("fundamentals backfill:", BACKFILL or "NOT FOUND (fundamentals inputs will be zero)")
print("CSE history beyond one year:", CSE_HISTORY or "NOT FOUND (CSE uses the one-year endpoint window only)")

## Smoke run

Five CSE names, one horizon, one epoch: catches data, shape and device problems in minutes.

In [ ]:
from experiments.train_hybrid_heads import main

args = ["--out", str(OUT / "smoke"), "--cse-limit", "5", "--horizons", "5",
        "--epochs", "1", "--walk-forward-points", "20", "--timesfm-generation", TIMESFM_GENERATION,
        "--split", "per-class"]
if BACKFILL:
    args += ["--backfill", str(BACKFILL)]
if CSE_HISTORY:
    args += ["--cse-history", str(CSE_HISTORY)]
main(args)

## Full training

Every CSE security with >= 200 sessions, the top 50 Binance USDT pairs, and the US universe.
The base forecasts (walk-forward TimesFM passes) dominate the run time.

In [ ]:
args = ["--out", str(OUT), "--timesfm-generation", TIMESFM_GENERATION, "--split", "per-class"]
if BACKFILL:
    args += ["--backfill", str(BACKFILL)]
if CSE_HISTORY:
    args += ["--cse-history", str(CSE_HISTORY)]
main(args)

## Holdout result (RQ1 input)

Pinball loss on the time-held-out final 20% of dates: base TimesFM vs the hybrid, overall and
per asset class. Lower is better. A hybrid that does not beat its base on a class is reported
as such, not hidden.

In [ ]:
from evaluation.rq1_table import holdout_table

# One row per (horizon, asset class). `note` flags a small holdout and the CSE one-year limit on the
# row itself; read it before quoting a CSE number.
table = holdout_table(OUT)
table


## Bring the heads home

Then, on the local machine:

```bash
python -m forecasting.serving --install hybrid_heads.zip
```

That registers `hybrid_h5` / `hybrid_h21` in the model registry, so responses carry a real
`model_version`. Serving also needs TimesFM's weights cached locally (~821 MB, once):
`huggingface-cli download google/timesfm-2.5-200m-pytorch`. Until then the analysis pipeline
uses a labelled empirical band and `/forecast` answers 503 with that reason.

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(OUT / "hybrid_heads.zip"))